# 01. Workspace, Dependencies, and Reproducibility

        Noa Yarin Levy · Jimi Halawi · Ofir Atias

        **Purpose.** Verify that the repository, data files, Python environment, and required packages are available before any analysis begins.

        **Inputs.** The project directory, `requirements.txt`, and the files under `data/`.

        **Outputs.** Package versions, data-file inventory, cryptographic hashes, and pre-run checks.

        **Research role.** This is the reproducibility foundation shared by RQ1-RQ4.

        The notebook is self-contained: it initializes the project paths, defines the needed code, runs the relevant step, and explains the saved results.

## Project scope and research questions

The project studies 37,700 GitHub developers and 289,003 mutual-follow links. It asks whether network structure helps explain and predict the binary Web/ML field label. The analysis is descriptive and does not claim causality.

1. **RQ1:** Can graph features classify developer field without test-label leakage?
2. **RQ2:** What does content add to network structure, and what does their hybrid add?
3. **RQ3:** Can hidden static edges be reconstructed, and does same-field reconstruction differ from cross-field reconstruction?
4. **RQ4:** Are homophily and modularity unusual under simple null models?

## Portable project initialization

This cell finds the project root, moves execution there, and adds `src` to the import path. It also sets deterministic defaults. Expected result: the absolute project-root path.

In [1]:
from pathlib import Path
import os
import sys
import warnings

def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "data").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate a project root containing requirements.txt and data/")

ROOT = _find_project_root(Path.cwd())
os.chdir(ROOT)
SRC_DIR = ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
os.environ.setdefault("PYTHONHASHSEED", "0")
warnings.filterwarnings("ignore", category=FutureWarning)
__name__ = "research_notebook_01"
print(f"Project root: {ROOT}")

Project root: C:\Users\Jimi\Desktop\github-social-network


**Result.** The printed path must end in `github-social-network`. This confirms that all relative paths point to the submitted project folder.

## Conditional dependency check

The cell checks every import required by `requirements.txt`. It runs `pip install -r requirements.txt` only when a dependency is missing, so a clean environment is repaired while repeated runs remain fast. Expected result: `All required imports are available`, or the package-installation log.

In [2]:
import importlib.util
import subprocess

required_imports = {
    "pandas": "pandas", "numpy": "numpy", "networkx": "networkx",
    "matplotlib": "matplotlib", "sklearn": "scikit-learn",
    "seaborn": "seaborn", "community": "python-louvain",
    "scipy": "scipy", "gensim": "gensim", "powerlaw": "powerlaw",
    "nbformat": "nbformat", "nbconvert": "nbconvert", "docx": "python-docx",
    "pypdf": "pypdf", "bidi": "python-bidi", "reportlab": "reportlab",
}
missing = [package for module, package in required_imports.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements.txt")])
else:
    print("All required imports are available")

All required imports are available


**Result.** The availability message confirms that no manual package correction is required. Conditional installation makes the notebook portable without reinstalling the environment on every run.

## Package versions, data files, and hashes

This cell records installed package versions, file sizes, and SHA-256 hashes for the MUSAE inputs. Hashes make data provenance auditable: a third party can distinguish a changed dataset from a changed algorithm. Expected result: one version row per major dependency and one inventory row per MUSAE data file.

In [3]:
import hashlib
import importlib.metadata as metadata
import pandas as pd

packages = ["pandas", "numpy", "networkx", "matplotlib", "scikit-learn", "seaborn", "python-louvain", "gensim", "powerlaw"]
versions = pd.DataFrame({"package": packages, "version": [metadata.version(p) for p in packages]})
display(versions)

data_rows = []
for path in sorted((ROOT / "data").glob("musae_git_*")):
    data_rows.append({
        "file": path.name,
        "bytes": path.stat().st_size,
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
    })
data_inventory = pd.DataFrame(data_rows)
display(data_inventory)

,package,version
0,pandas,2.2.3
1,numpy,1.26.4
2,networkx,3.6.1
3,matplotlib,3.10.0
4,scikit-learn,1.6.0
5,seaborn,0.13.2
6,python-louvain,0.16
7,gensim,4.4.0
8,powerlaw,2.0.0


,file,bytes,sha256
0,musae_git_edges.csv,3595143,d0d6a4f16e6ce841ddcdd0ff3aac49a9d4da157ba0dea8...
1,musae_git_features.json,4380176,47761f93f839d84b213e9f2fa43c056761734b295d6b2e...
2,musae_git_target.csv,714229,99ab263ad81c342fea4cd141c79f3094e35a670af458f3...


**Result.** The version and hash tables capture the exact execution context. They are especially useful when investigating small numerical differences in Word2Vec, linear algebra, or other stochastic and platform-dependent routines.

## Summary

        **Completed.** Located the repository, checked or installed dependencies, and recorded package versions and data hashes.

        **Key findings.** The environment is complete and all three MUSAE inputs are directly accessible.

        **Workflow connection.** The validated inputs are ready for graph construction and every research question.